# HWP 텍스트 추출 테스트

OLE 기반 `.hwp` 파일에서 `BodyText` stream을 읽고, 압축 여부를 확인한 뒤 `PARA_TEXT` record만 UTF-16LE로 디코딩합니다.

HWP 내부 이미지/스타일/필드 제어 정보가 한자처럼 잘못 보이는 경우가 있어, `remove_hwp_garbage()`에서 packed ASCII 형태의 HWP artifact를 제거합니다.

주의: 이 코드는 본문 텍스트 추출용입니다. 표의 행/열 구조 복원과 이미지 OCR은 별도 파이프라인으로 처리해야 합니다.


## Large File Note

원본 HWP/PDF와 parsing output(JSONL/XLSX/CSV)은 용량이 크기 때문에 GitHub에 포함하지 않습니다. 실행 전 공유 Drive 또는 로컬/VM 디스크에 `data/`와 `outputs/` 구조를 먼저 배치합니다.


In [1]:
from pathlib import Path
from collections import Counter
import importlib.util
import re
import subprocess
import sys
import zlib


def ensure_package(import_name: str, package_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name or import_name])


ensure_package("olefile")
ensure_package("pandas")
ensure_package("IPython", "ipython")

from IPython.display import display
import olefile
import pandas as pd


# Large input files are intentionally not committed. Place data/original_data_list on local disk before running.
PROJECT_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r"C:\Users\yoosy\Desktop\codeit\project_2nd"),
    Path(r"C:\Users\goodm\Desktop\codeit\project_2nd"),
]

def resolve_project_dir(required=("data",)):
    for path in PROJECT_DIR_CANDIDATES:
        if all((path / item).exists() for item in required):
            return path
    checked = "\n".join(str(path) for path in PROJECT_DIR_CANDIDATES)
    raise FileNotFoundError(f"Project directory not found. Checked:\n{checked}")


def resolve_repo_dir(project_dir):
    for path in [project_dir, project_dir / "DB_RAG_Codeit_Project", Path.cwd(), Path.cwd().parent]:
        if (path / "requirements.txt").exists():
            return path
    return project_dir

PROJECT_DIR = resolve_project_dir(required=("data",))
REPO_DIR = resolve_repo_dir(PROJECT_DIR)
HWPTAG_PARA_TEXT = 67
KEEP_HANJA_RUNS = {
    "甲乙", "甲乙丙", "甲乙丙丁",
    "案內", "內外", "共有",
    "未定", "無償", "有償",
}


def is_packed_ascii_garbage(text: str) -> bool:
    """HWP control/object bytes decoded as CJK-like characters often become packed ASCII."""
    if len(text) < 2:
        return False

    raw = b"".join(ch.encode("utf-16le", errors="ignore") for ch in text)
    if not raw:
        return False

    printable = sum(32 <= b <= 126 for b in raw)
    ascii_letters = sum((65 <= b <= 90) or (97 <= b <= 122) for b in raw)

    return printable / len(raw) >= 0.8 and ascii_letters >= len(text)


def remove_hwp_garbage(text: str) -> str:
    """Remove common HWP inline-control artifacts while keeping normal Korean text."""

    # Remove control characters first. Some HWP artifacts appear as "螨\x00ȃ\x00".
    text = text.replace("\x00", "")
    text = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f]", " ", text)

    def replace_cjk_run(match):
        token = match.group(0)
        if token in KEEP_HANJA_RUNS:
            return token
        return " " if is_packed_ascii_garbage(token) else token

    # Examples removed by this rule: 捤獥, 汤捯, 湰灧, 氠瑢, 桤灧, 楴䵴
    text = re.sub(r"[\u4e00-\u9fff]{2,}", replace_cjk_run, text)

    # Examples removed by this rule: 螨ȃ, 砼ȃ, 牠ȃ, 沈ȃ
    text = re.sub(r"[\u4e00-\u9fff][\u0200-\u02ff]", " ", text)

    cleaned_lines = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if not line:
            continue

        # Drop pure artifact lines. Keep lines that contain Korean, Latin, or digits.
        if not re.search(r"[가-힣A-Za-z0-9]", line):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


HANJA_TOKEN_RE = re.compile(r"[\u4e00-\u9fff]+")
MEANING_CONTEXT_RE = re.compile(r"[가-힣A-Za-z0-9]")
ARTIFACT_CONTEXT_RE = re.compile(r"[\u0200-\u02ff\u0800-\u08ff\u2500-\u257f†╦]")


def packed_ascii_score(text: str) -> float:
    raw = b"".join(ch.encode("utf-16le", errors="ignore") for ch in text)
    if not raw:
        return 0.0
    printable = sum(32 <= b <= 126 for b in raw)
    return round(printable / len(raw), 3)


def compact_context(text: str, start: int, end: int, window: int = 45) -> str:
    left = max(0, start - window)
    right = min(len(text), end + window)
    context = text[left:right].replace("\x00", "")
    context = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f]", " ", context)
    context = re.sub(r"\s+", " ", context).strip()
    return context


def has_inline_hwp_artifact(raw_text: str, token_end: int) -> bool:
    tail = raw_text[token_end:token_end + 8].replace("\x00", "")
    tail = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f]", "", tail)
    return bool(tail and re.match(r"[\u0200-\u02ff]", tail))


def has_artifact_context(context: str, token: str) -> bool:
    nearby = context.replace(token, "")
    if ARTIFACT_CONTEXT_RE.search(nearby):
        return True

    for match in HANJA_TOKEN_RE.finditer(nearby):
        candidate = match.group(0)
        if len(candidate) >= 2 and is_packed_ascii_garbage(candidate):
            return True

    return False


def find_hanja_exception_candidates(raw_text: str, clean_text: str, top_n: int = 80) -> pd.DataFrame:
    """Return CJK tokens with evidence for keep/remove review.

    This does not automatically expand KEEP_HANJA_RUNS. The output is a review table
    for deciding which normal Hanja expressions should be preserved.
    """
    matches = list(HANJA_TOKEN_RE.finditer(raw_text))
    counts = Counter(match.group(0) for match in matches)
    first_match = {}
    for match in matches:
        first_match.setdefault(match.group(0), match)

    rows = []
    for token, count in counts.most_common(top_n):
        match = first_match[token]
        context = compact_context(raw_text, match.start(), match.end())
        has_meaning_context = bool(MEANING_CONTEXT_RE.search(context.replace(token, "")))
        survives_clean = token in clean_text
        is_packed = is_packed_ascii_garbage(token)
        inline_artifact = has_inline_hwp_artifact(raw_text, match.end())
        artifact_context = has_artifact_context(context, token)

        if token in KEEP_HANJA_RUNS:
            action = "keep_confirmed"
            reason = "수동 승인된 보존 표현"
        elif is_packed or inline_artifact:
            action = "remove_artifact"
            reason = "HWP 제어 정보가 한자처럼 디코딩된 패턴"
        elif artifact_context:
            action = "review_artifact_context"
            reason = "clean 후 남았지만 주변 artifact 패턴이 있어 검토 필요"
        elif not survives_clean and has_meaning_context:
            action = "review_removed"
            reason = "clean 후 사라졌지만 주변 문맥이 있어 검토 필요"
        elif survives_clean and has_meaning_context:
            action = "keep_candidate"
            reason = "clean 후 보존되고 주변 문맥이 있음"
        elif survives_clean:
            action = "review_keep"
            reason = "보존되었지만 문맥 근거가 약함"
        else:
            action = "review_remove"
            reason = "문맥 근거가 약하고 clean 후 제거됨"

        rows.append(
            {
                "token": token,
                "count": count,
                "survives_clean": survives_clean,
                "packed_ascii_score": packed_ascii_score(token),
                "recommended_action": action,
                "reason": reason,
                "sample_context": context,
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    action_order = {
        "keep_confirmed": 0,
        "review_removed": 1,
        "review_artifact_context": 2,
        "keep_candidate": 3,
        "review_keep": 4,
        "remove_artifact": 5,
        "review_remove": 6,
    }
    df["_order"] = df["recommended_action"].map(action_order).fillna(99)
    df = df.sort_values(["_order", "count"], ascending=[True, False])
    return df.drop(columns=["_order"]).reset_index(drop=True)

def is_hwp_compressed(ole) -> bool:
    header = ole.openstream("FileHeader").read()
    return bool(header[36] & 1)


def decompress_hwp_stream(data: bytes, compressed: bool) -> bytes:
    if not compressed:
        return data

    try:
        return zlib.decompress(data, -15)
    except zlib.error:
        return zlib.decompress(data)


def iter_hwp_records(section_data: bytes):
    pos = 0
    size_total = len(section_data)

    while pos + 4 <= size_total:
        header = int.from_bytes(section_data[pos:pos + 4], "little")
        pos += 4

        tag_id = header & 0x3ff
        level = (header >> 10) & 0x3ff
        size = (header >> 20) & 0xfff

        if size == 0xfff:
            if pos + 4 > size_total:
                break
            size = int.from_bytes(section_data[pos:pos + 4], "little")
            pos += 4

        payload = section_data[pos:pos + size]
        pos += size

        yield tag_id, level, payload


def extract_hwp_text(hwp_path: str | Path) -> dict:
    hwp_path = Path(hwp_path)

    if not olefile.isOleFile(str(hwp_path)):
        raise ValueError(f"OLE 기반 HWP 파일이 아닙니다: {hwp_path}")

    ole = olefile.OleFileIO(str(hwp_path))
    compressed = is_hwp_compressed(ole)

    section_names = sorted(
        "/".join(item)
        for item in ole.listdir(streams=True, storages=False)
        if item and item[0] == "BodyText"
    )

    raw_texts = []
    para_text_record_count = 0

    for section_name in section_names:
        raw = ole.openstream(section_name).read()
        data = decompress_hwp_stream(raw, compressed)

        for tag_id, level, payload in iter_hwp_records(data):
            if tag_id == HWPTAG_PARA_TEXT:
                para_text_record_count += 1
                text = payload.decode("utf-16le", errors="ignore")
                if text.strip():
                    raw_texts.append(text)

    ole.close()

    raw_text = "\n".join(raw_texts)
    clean_text = remove_hwp_garbage(raw_text)

    return {
        "filename": hwp_path.name,
        "path": str(hwp_path),
        "compressed": compressed,
        "section_count": len(section_names),
        "para_text_record_count": para_text_record_count,
        "raw_char_len": len(raw_text),
        "clean_char_len": len(clean_text),
        "text": clean_text,
    }


def extract_hwp_text_only(hwp_path: str | Path) -> str:
    return extract_hwp_text(hwp_path)["text"]


## Raw vs Clean 비교

`extract_hwp_text_raw()`는 같은 HWP record 파싱 로직을 사용하되, artifact cleaning을 적용하지 않습니다.
사람이 비교하기 쉽도록 요약, raw 샘플, clean 샘플을 서로 다른 코드셀로 분리합니다.


In [2]:
def extract_hwp_text_raw(hwp_path: str | Path) -> dict:
    """Same HWP parsing logic as extract_hwp_text(), but without artifact cleaning."""
    hwp_path = Path(hwp_path)

    if not olefile.isOleFile(str(hwp_path)):
        raise ValueError(f"OLE 기반 HWP 파일이 아닙니다: {hwp_path}")

    ole = olefile.OleFileIO(str(hwp_path))
    compressed = is_hwp_compressed(ole)

    section_names = sorted(
        "/".join(item)
        for item in ole.listdir(streams=True, storages=False)
        if item and item[0] == "BodyText"
    )

    raw_texts = []
    para_text_record_count = 0

    for section_name in section_names:
        raw = ole.openstream(section_name).read()
        data = decompress_hwp_stream(raw, compressed)

        for tag_id, level, payload in iter_hwp_records(data):
            if tag_id == HWPTAG_PARA_TEXT:
                para_text_record_count += 1
                text = payload.decode("utf-16le", errors="ignore")
                if text.strip():
                    raw_texts.append(text)

    ole.close()

    raw_text = "\n".join(raw_texts)

    return {
        "filename": hwp_path.name,
        "path": str(hwp_path),
        "compressed": compressed,
        "section_count": len(section_names),
        "para_text_record_count": para_text_record_count,
        "raw_char_len": len(raw_text),
        "text": raw_text,
    }


In [3]:
PROJECT_DIR = resolve_project_dir(required=("data",))
DATA_DIR = PROJECT_DIR / "data" / "original_data_list"

hwp_files = sorted(DATA_DIR.rglob("*.hwp"))
if not hwp_files:
    raise FileNotFoundError(f"HWP 파일을 찾을 수 없습니다: {DATA_DIR}")

HWP_PATH = hwp_files[0]

raw_result = extract_hwp_text_raw(HWP_PATH)
clean_result = extract_hwp_text(HWP_PATH)

print("파일명:", HWP_PATH.name)
print("압축 여부:", clean_result["compressed"])
print("BodyText 섹션 수:", clean_result["section_count"])
print("PARA_TEXT record 수:", clean_result["para_text_record_count"])
print("raw 문자 수:", raw_result["raw_char_len"])
print("clean 문자 수:", clean_result["clean_char_len"])


파일명: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp
압축 여부: True
BodyText 섹션 수: 1
PARA_TEXT record 수: 5002
raw 문자 수: 113435
clean 문자 수: 102214


## Raw 추출 샘플

artifact 제거 전 원문 일부입니다.


In [4]:
print(raw_result["text"][:1200])


捤獥    汤捯    湰灧    
氠瑢    
2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업
(복수의결권주식, 스톡옵션, 성과조건부주식) -
제안요청서
2024. 03.桤灧    
漠杳    
氠瑢    
桤灧    
氠瑢    
목  차
 1. 추진개요	螨 ȃ   	 3
 2. 추진방안	螨 ȃ   	 5
 3. 추진내용	螨 ȃ   	 9
 4. 제안요청내용	砼 ȃ   	 24
 5. 입찰관련사항	砼 ȃ   	 78
 6. 제안서작성요령	牠 ȃ   	 82
 7. 별지서식 및 붙임	沈 ȃ   	 94
氠瑢    
Ⅰ. 추진개요
氠瑢    
1
추진배경 및 방향
□ 「벤처기업육성에 관한 특별조치법」(이하 ‘벤처기업법’) 복수의결권주식, 스톡옵션(주식매수선택권), 성과조건부주식교부계약(RS) 등의 기능 고도화 및 이관, 신규 구축업무를 本 과업에서 추진
  ⚬(복수의결권주식*) 벤처기업법 제16조의11에 따라 발행된 복수의결권주식 보고 업무처리 시스템 구축
      * 복수의결권이란? 모든 주주는 1주당 하나의 의결권을 갖는 주주평등원칙(상법)과 별도로 비상장 벤처기업 창업자에게만 1주당 최대 10배의 의결권 행사를 부여하는 제도 
  ⚬(스톡옵션*) 벤처기업법 제16조의3에 따라 부여된 벤처기업 스톡옵션 부여, 취소·철회 신고 및 업무시스템 구축
     * 스톡옵션이란? 비상장 벤처기업 임·직원 및 기업 성장에 기여한 자에게 미리 정한 가격(행사가격)으로 신주를 인수하거나 자기의 주식을 매수할 수 있는 권리 혹은 주식의 시가와 행사가격의 차액을 청구할 수 있는 권리를 부여하는 제도
 -중소벤처24 內 스톡옵션 신청 기능 이관을 추진하며, 과거 데이터 정제 및 적재 등을 통한 데이터 이관 추진
  ⚬(성과조건부주식*) 벤처기업법 제16조의19에 따라 성과조건부주식교부계약의 신고 및 업무처리 시스템 구축
     * 성과조건부주식이란? 비상장 벤처기업 임ㆍ직원 및 특정인에게 자기주식 부여를 

## Clean 추출 샘플

`remove_hwp_garbage()` 적용 후 원문 일부입니다.


In [5]:
print(clean_result["text"][:1200])


2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업
(복수의결권주식, 스톡옵션, 성과조건부주식) -
제안요청서
2024. 03.
목 차
1. 추진개요 3
2. 추진방안 5
3. 추진내용 9
4. 제안요청내용 24
5. 입찰관련사항 78
6. 제안서작성요령 82
7. 별지서식 및 붙임 94
Ⅰ. 추진개요
1
추진배경 및 방향
□ 「벤처기업육성에 관한 특별조치법」(이하 ‘벤처기업법’) 복수의결권주식, 스톡옵션(주식매수선택권), 성과조건부주식교부계약(RS) 등의 기능 고도화 및 이관, 신규 구축업무를 本 과업에서 추진
⚬ (복수의결권주식*) 벤처기업법 제16조의11에 따라 발행된 복수의결권주식 보고 업무처리 시스템 구축
* 복수의결권이란? 모든 주주는 1주당 하나의 의결권을 갖는 주주평등원칙(상법)과 별도로 비상장 벤처기업 창업자에게만 1주당 최대 10배의 의결권 행사를 부여하는 제도
⚬ (스톡옵션*) 벤처기업법 제16조의3에 따라 부여된 벤처기업 스톡옵션 부여, 취소·철회 신고 및 업무시스템 구축
* 스톡옵션이란? 비상장 벤처기업 임·직원 및 기업 성장에 기여한 자에게 미리 정한 가격(행사가격)으로 신주를 인수하거나 자기의 주식을 매수할 수 있는 권리 혹은 주식의 시가와 행사가격의 차액을 청구할 수 있는 권리를 부여하는 제도
- 중소벤처24 內 스톡옵션 신청 기능 이관을 추진하며, 과거 데이터 정제 및 적재 등을 통한 데이터 이관 추진
⚬ (성과조건부주식*) 벤처기업법 제16조의19에 따라 성과조건부주식교부계약의 신고 및 업무처리 시스템 구축
* 성과조건부주식이란? 비상장 벤처기업 임ㆍ직원 및 특정인에게 자기주식 부여를 약속하되, 일정 재직기간 및 조건을 충족해야 주식이 부여 대상자에게 귀속되도록 제한을 둔 주식
□ 복수의결권주식 발행 보고 및 스톡옵션 신고기능은 기존 시스템 내 벤처확인 이력정보와 신고자(벤처기업)의 정보와 연동
2
사업개요
□ (사 업 명) 2024년 「벤처확인종합관리시스템 기능 고도화」 용역사업
- 복수의결권주식, 스톡옵션, 성과조건부주

## Artifact 제거 확인

raw 샘플과 clean 샘플을 눈으로 확인한 뒤, 주요 HWP artifact token이 실제로 제거됐는지 정량적으로 확인합니다.


In [6]:
tokens = ["捤獥", "氠瑢", "桤灧", "漠杳", "楴䵴", "螨ȃ", "砼ȃ", "牠ȃ", "沈ȃ"]

print("\n--- artifact token check ---")
for token in tokens:
    print(f"{token} | raw={token in raw_result['text']} | clean={token in clean_result['text']}")



--- artifact token check ---
捤獥 | raw=True | clean=False
氠瑢 | raw=True | clean=False
桤灧 | raw=True | clean=False
漠杳 | raw=True | clean=False
楴䵴 | raw=False | clean=False
螨ȃ | raw=False | clean=False
砼ȃ | raw=False | clean=False
牠ȃ | raw=False | clean=False
沈ȃ | raw=False | clean=False


## 한자 예외 후보 추출

HWP artifact 제거 과정에서 정상 한자 표현이 같이 제거될 수 있으므로, raw 텍스트의 CJK token을 수집하고 `보존 후보`, `검토 필요`, `제거 후보`로 분류합니다.
최종 `KEEP_HANJA_RUNS`는 이 표를 사람이 검토한 뒤에만 확정합니다.


In [ ]:
hanja_candidates = find_hanja_exception_candidates(raw_result["text"], clean_result["text"], top_n=80)

print("--- recommended_action counts ---")
print(hanja_candidates["recommended_action"].value_counts().to_string())

print("\n--- top hanja exception candidates ---")
columns = [
    "token",
    "count",
    "survives_clean",
    "packed_ascii_score",
    "recommended_action",
    "reason",
    "sample_context",
]
print(hanja_candidates[columns].head(5).to_string(index=False))


## 200개 pilot 한자 예외 후보 추출

문서 1개 기준 후보는 샘플 검증용입니다. 실제 보존/제거 판단은 여러 문서에서의 `count`, `doc_count`, `artifact_context_count`를 보고 결정합니다.

아래 셀은 실행 시간이 걸릴 수 있으므로, 데이터가 있는 작업 컴퓨터에서만 실행합니다.


In [8]:
PILOT_LIMIT = 200
TOP_N_PER_DOC = 120
PROJECT_DIR = resolve_project_dir(required=("data",))
DATA_DIR = PROJECT_DIR / "data" / "original_data_list"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
HANJA_CANDIDATE_CSV = OUTPUTS_DIR / "hanja_exception_candidates.csv"


def build_corpus_hanja_candidates(hwp_paths, limit=200, top_n_per_doc=120) -> pd.DataFrame:
    rows = []
    errors = []

    selected_paths = list(hwp_paths[:limit]) if limit else list(hwp_paths)

    for idx, hwp_path in enumerate(selected_paths, start=1):
        try:
            raw_doc = extract_hwp_text_raw(hwp_path)
            clean_doc = extract_hwp_text(hwp_path)
            candidates = find_hanja_exception_candidates(
                raw_doc["text"],
                clean_doc["text"],
                top_n=top_n_per_doc,
            )

            if candidates.empty:
                continue

            candidates = candidates.copy()
            candidates["source_file"] = hwp_path.name
            candidates["source_path"] = str(hwp_path)
            rows.append(candidates)

            if idx % 20 == 0:
                print(f"processed {idx}/{len(selected_paths)} files")
        except Exception as exc:
            errors.append({"source_file": hwp_path.name, "error": repr(exc)})
            print(f"[WARN] failed: {hwp_path.name} | {exc}")

    if not rows:
        print("candidate row가 없습니다.")
        return pd.DataFrame()

    detail_df = pd.concat(rows, ignore_index=True)

    summary_df = (
        detail_df
        .groupby("token", as_index=False)
        .agg(
            count=("count", "sum"),
            doc_count=("source_file", "nunique"),
            survives_clean_count=("survives_clean", "sum"),
            packed_ascii_score_avg=("packed_ascii_score", "mean"),
            remove_artifact_count=("recommended_action", lambda s: (s == "remove_artifact").sum()),
            artifact_context_count=("recommended_action", lambda s: (s == "review_artifact_context").sum()),
            keep_candidate_count=("recommended_action", lambda s: (s == "keep_candidate").sum()),
            review_removed_count=("recommended_action", lambda s: (s == "review_removed").sum()),
            sample_context=("sample_context", lambda s: " || ".join(s.dropna().astype(str).head(3))),
        )
    )

    def decide_action(row):
        if row["remove_artifact_count"] >= max(2, row["doc_count"] * 0.6):
            return "remove_artifact"
        if row["artifact_context_count"] >= max(2, row["doc_count"] * 0.4):
            return "review_artifact_context"
        if row["doc_count"] >= 3 and row["keep_candidate_count"] >= 2:
            return "keep_candidate"
        if row["review_removed_count"] > 0:
            return "review_removed"
        return "review"

    summary_df["recommended_action"] = summary_df.apply(decide_action, axis=1)
    summary_df["packed_ascii_score_avg"] = summary_df["packed_ascii_score_avg"].round(3)

    action_order = {
        "keep_candidate": 0,
        "review_removed": 1,
        "review_artifact_context": 2,
        "review": 3,
        "remove_artifact": 4,
    }
    summary_df["_order"] = summary_df["recommended_action"].map(action_order).fillna(99)
    summary_df = summary_df.sort_values(
        ["_order", "doc_count", "count"],
        ascending=[True, False, False],
    ).drop(columns=["_order"])

    OUTPUTS_DIR.mkdir(exist_ok=True)
    summary_df.to_csv(HANJA_CANDIDATE_CSV, index=False, encoding="utf-8-sig")

    print(f"대상 파일 수: {len(selected_paths)}")
    print(f"성공 문서 후보 row 수: {len(detail_df)}")
    print(f"실패 파일 수: {len(errors)}")
    print(f"저장 위치: {HANJA_CANDIDATE_CSV}")
    if errors:
        print("\n--- failed files preview ---")
        print(pd.DataFrame(errors).head(20).to_string(index=False))

    return summary_df


all_hwp_files = sorted(DATA_DIR.rglob("*.hwp"))
hanja_corpus_candidates = build_corpus_hanja_candidates(
    all_hwp_files,
    limit=PILOT_LIMIT,
    top_n_per_doc=TOP_N_PER_DOC,
)

display_cols = [
    "token",
    "count",
    "doc_count",
    "survives_clean_count",
    "packed_ascii_score_avg",
    "remove_artifact_count",
    "artifact_context_count",
    "keep_candidate_count",
    "recommended_action",
    "sample_context",
]

hanja_corpus_candidates[display_cols].head(50)


processed 20/200 files


processed 40/200 files


processed 60/200 files


processed 80/200 files


processed 100/200 files


...

processed 160/200 files


processed 180/200 files


processed 200/200 files


대상 파일 수: 200
성공 문서 후보 row 수: 5372
실패 파일 수: 0
저장 위치: C:\Users\goodm\Desktop\codeit\project_2nd\outputs\hanja_exception_candidates.csv


,token,count,doc_count,survives_clean_count,packed_ascii_score_avg,remove_artifact_count,artifact_context_count,keep_candidate_count,recommended_action,sample_context
126,同,54,46,46,0.50,0,1,45,keep_candidate,"호에 관한 법률’ 제2조2호 개인정보 ⑩ ‘보안업무규정’ 제4조의 비밀, 同시행규칙..."
70,內,35,18,18,1.00,0,0,18,keep_candidate,가격의 차액을 청구할 수 있는 권리를 부여하는 제도 - 중소벤처24 內 스톡옵션 신...
1181,現,32,18,18,0.50,0,7,11,keep_candidate,"고도화 수행 (복수의결권, 스톡옵션, 성과조건부주식) 운영 유지관리 업체 現 시스템..."
448,必,20,16,16,0.50,0,2,14,keep_candidate,(천원) 발주처 책임 전문가 증빙 쪽 1 분야별/ 국내/해외 구분 必 2 3 ※ 유...
2046,非,19,13,13,0.50,0,0,13,keep_candidate,"승인 후 반출 전산장비 망간 혼용 금지, 부득이한 경우 시스템 포맷 후 사용 非인가..."
87,前,21,11,11,1.00,0,0,11,keep_candidate,징역 또는 5천만원 이하 벌금 □ (비상장 벤처기업) 복수의결권주식 발행보고 前 비...
71,全,15,9,9,1.00,0,2,7,keep_candidate,"채널(포털, 앱, 전화, 이메일, 방문 등)의 온․오프라인 기업애로 접수․상담 全 ..."
1394,私人,9,9,9,0.50,0,0,9,keep_candidate,"한 항의, 대상 및 내용이 불명확하거나 근거 없는 비방으로 판단되는 경우, 사인(私..."
691,本,7,7,7,1.00,0,1,6,keep_candidate,"권), 성과조건부주식교부계약(RS) 등의 기능 고도화 및 이관, 신규 구축업무를 本..."
1508,糨,11,6,3,0.50,3,0,3,keep_candidate,안내 1. 입찰 및 용역사 선정 방법 惈ȃ 43 2. 입찰참가자격 糨ȃ 43 3. ...


## review_artifact_context 전체 확인

`head(50)` 출력에서는 정렬 순서 때문에 일부만 보일 수 있습니다. 아래 셀은 `review_artifact_context`로 자동 분류된 token 전체를 별도로 확인합니다.


In [9]:
review_artifact_context_full = hanja_corpus_candidates[
    hanja_corpus_candidates["recommended_action"] == "review_artifact_context"
].copy()

print("review_artifact_context token 수:", len(review_artifact_context_full))
review_artifact_context_full[display_cols]


review_artifact_context token 수: 14


,token,count,doc_count,survives_clean_count,packed_ascii_score_avg,remove_artifact_count,artifact_context_count,keep_candidate_count,recommended_action,sample_context
932,浵,443,132,132,1.0,0,132,0,review_artifact_context,수행을 위해 필요한 지원 사항 및 방안에 대한 요구사항을 기술 9 합 계 浵╦ 11...
883,汫,666,85,85,1.0,0,85,0,review_artifact_context,데이터를 작성 및 제출 ○ SW사업정보 데이터 작성 및 제출에 관한 사항은 汫╨ w...
774,楴,1060,82,79,1.0,1,59,21,review_artifact_context,"에서 보다 창의적인 방법으로 요구사항을 해석하고, 정제하여 사업에 반영해야 함 楴䵴..."
662,普,120,60,60,1.0,0,60,0,review_artifact_context,포함한 문서 [별표 2] 소프트웨어 보안약점 기준 소프트웨어 보안약점 기준 †普 湯...
928,浫,215,19,18,1.0,1,18,0,review_artifact_context,및 다운로드 기능을 제공하여야 함 o 카탈로거를 통해 생성된 歭扯 浫╢ 장면전환 프...
642,新,14,7,6,0.5,1,3,3,review_artifact_context,"가동중단 횟수가 3회 이상 발생하는 경우, 발주기관의 요구에 따라 동급 이상의 新 ..."
1123,牦,66,4,4,1.0,0,4,0,review_artifact_context,瑢 桤灧 Ⅰ. 사업개요 1. 사업안내 菠ȃ 牦╸ 2 牦x 2. 추진배경 및 필요성 ...
1676,舊,6,4,4,0.0,0,2,2,review_artifact_context,입M/M : 개인별 Man Month를 기술하며 월 미만은 절사 6. 비고 : 舊)...
1762,蕀,30,3,3,0.5,0,3,0,review_artifact_context,氠瑢 氠瑢 목 차 Ⅰ. 사업개요 1. 사업개요 蕀ă 1 2. 추진배경 蕀ă 1 3....
998,潣,6,3,1,1.0,0,3,0,review_artifact_context,031-8075-4893 桤灧 목 ـĀ ྠĀ 차 潣╴ Ⅰ. 사업개요 騴ȃ 1 1. ...


## review_artifact_context 문서별 요약

`review_artifact_context` token이 실제로 어떤 파일에서 얼마나 나왔는지 다시 수집합니다. 이 결과는 원문을 직접 열어 확인할 문서를 찾기 위한 검토용입니다.


In [10]:
REVIEW_SOURCE_CSV = OUTPUTS_DIR / "review_artifact_context_sources.csv"


def collect_review_artifact_context_sources(hwp_paths, target_tokens, limit=200, top_n_per_doc=120) -> pd.DataFrame:
    rows = []
    selected_paths = list(hwp_paths[:limit]) if limit else list(hwp_paths)

    for idx, hwp_path in enumerate(selected_paths, start=1):
        raw_doc = extract_hwp_text_raw(hwp_path)
        clean_doc = extract_hwp_text(hwp_path)
        candidates = find_hanja_exception_candidates(
            raw_doc["text"],
            clean_doc["text"],
            top_n=top_n_per_doc,
        )

        if candidates.empty:
            continue

        candidates = candidates[candidates["token"].isin(target_tokens)].copy()
        if candidates.empty:
            continue

        candidates["source_file"] = hwp_path.name
        candidates["source_path"] = str(hwp_path)
        rows.append(candidates)

        if idx % 50 == 0:
            print(f"review source scan {idx}/{len(selected_paths)} files")

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


review_tokens = review_artifact_context_full["token"].tolist()
review_artifact_detail = collect_review_artifact_context_sources(
    all_hwp_files,
    target_tokens=review_tokens,
    limit=PILOT_LIMIT,
    top_n_per_doc=TOP_N_PER_DOC,
)

if review_artifact_detail.empty:
    review_artifact_detail = pd.DataFrame(columns=[
        "token", "count", "source_file", "source_path", "sample_context"
    ])

review_artifact_detail.to_csv(REVIEW_SOURCE_CSV, index=False, encoding="utf-8-sig")

review_doc_summary = (
    review_artifact_detail
    .groupby("source_file", as_index=False)
    .agg(
        total_occurrences=("count", "sum"),
        unique_token_count=("token", "nunique"),
        tokens=("token", lambda s: ", ".join(sorted(set(s.astype(str))))),
    )
    .sort_values(["unique_token_count", "total_occurrences", "source_file"], ascending=[False, False, True])
)

review_token_doc_counts = (
    review_artifact_detail
    .groupby(["token", "source_file"], as_index=False)
    .agg(
        token_occurrences=("count", "sum"),
        sample_context=("sample_context", "first"),
    )
    .sort_values(["token", "token_occurrences", "source_file"], ascending=[True, False, True])
)

print("전체 후보 token 수:", len(hanja_corpus_candidates))
print("review_artifact_context token 수:", len(review_tokens))
print("review_artifact_context 포함 문서 수:", review_artifact_detail["source_file"].nunique())
print("review_artifact_context 상세 row 수:", len(review_artifact_detail))
print("저장 위치:", REVIEW_SOURCE_CSV)

print("\\n--- token별 포함 문서 수/발생 수 ---")
print(
    review_artifact_detail
    .groupby("token", as_index=False)
    .agg(
        total_occurrences=("count", "sum"),
        doc_count=("source_file", "nunique"),
    )
    .sort_values(["doc_count", "total_occurrences"], ascending=[False, False])
    .to_string(index=False)
)

print("\\n--- 문서별 발생 요약 상위 80개 ---")
print(review_doc_summary.head(80).to_string(index=False))

review_token_doc_counts.head(120)


review source scan 50/200 files


review source scan 100/200 files


review source scan 150/200 files


review source scan 200/200 files
전체 후보 token 수: 2081
review_artifact_context token 수: 14
review_artifact_context 포함 문서 수: 187
review_artifact_context 상세 row 수: 410
저장 위치: C:\Users\goodm\Desktop\codeit\project_2nd\outputs\review_artifact_context_sources.csv
\n--- token별 포함 문서 수/발생 수 ---
token  total_occurrences  doc_count
    浵                443        132
    汫                666         85
    楴               1060         82
    普                120         60
    浫                215         19
    新                 14          7
    牦                 66          4
    舊                  6          4
    蕀                 30          3
    潣                  6          3
    沤                  3          3
    爔                  3          3
    遠                  3          3
    浥                 11          2
\n--- 문서별 발생 요약 상위 80개 ---
                                           source_file  total_occurrences  unique_token_count              tokens
          경기도 군포시_(긴급)군포 AI기반 쓰레

,token,source_file,token_occurrences,sample_context
5,新,나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp,6,구현을 위한 효율적이고 체계적인 설비 온라인 시스템 구축 ○ 기술원의 新공정관리시스...
3,新,"국민연금공단_(재공고)공동 연계급여정보시스템 재구축(1차, 2차).hwp",3,1. 하도급계획 적정성 汤捯 Ⅷ. 기타사항 ɘĀ 1. 舊·新 시스템 간 운영 전환 ...
0,新,(재)대전일자리경제진흥원_대전 소상공인 상권분석서비스 시스템 고도.hwp,1,"가동중단 횟수가 3회 이상 발생하는 경우, 발주기관의 요구에 따라 동급 이상의 新 ..."
1,新,국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp,1,"추진 등을 통한 관련 湯湷 산업 생태계 활성화 ㅇ 자율차, 드론 등 新산업의 기반 ..."
2,新,국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역.hwp,1,"추진 등을 통한 관련 湯湷 산업 생태계 활성화 ㅇ 자율차, 드론 등 新산업의 기반 ..."
...,...,...,...,...
131,楴,국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp,6,임 9】 참조 氠瑢 Ⅴ 제안요청 내용 1. 요구사항 총괄표 楴䵴 氠瑢 구 분 설 명...
82,楴,강원특별자치도 영월군_한반도농협 농산물산지유통센터 스마트APC 시스.hwp,5,(인) 한반도농업협동조합 귀하 <붙임 3-1> 楴䵴 사 용 인 감 계 氠瑢 사용인감...
88,楴,경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp,5,박종일 2024. 6. 桤灧 桤灧 湯湷 楴䵴 목 차 Ⅰ. 사업의 개요 萄ȃ 1 Ⅱ....
138,楴,남서울대학교_{대학혁신지원사업] 남서울대학교 대학원 학사시스템 구.hwp,5,술자 초급기술자 ※ 기술자의 구분은 [붙임 2]에 의함. [붙임2] 楴䵴 기술자의 ...


## 원문 검색 기반 검증 아이디어

한컴오피스에서 원문을 열어 `Ctrl+F`로 검색했는데 token이 없다면, raw parser가 HWP 제어 정보를 한자처럼 잘못 디코딩했을 가능성이 높습니다.

정석 검증은 `review_artifact_context` token이 포함된 원본만 HWPX 또는 PDF로 변환한 뒤, 독립 추출기로 다시 텍스트를 뽑아 token 존재 여부를 비교하는 방식입니다. 독립 추출 텍스트에도 없고 raw parser에만 있으면 제거 후보로 확정합니다.


## PDF 기반 artifact 검증 workflow

`review_artifact_context` 14개 token 전체를 포함합니다. `新`, `舊`처럼 정상 한자라고 알고 있는 token도 positive control로 일부러 포함해, PDF 검증 방식이 정상 한자와 artifact를 구분하는지 확인합니다.

검증은 두 단계로 나눕니다.

1. 대표 문서 검증: token별 대표 문서 1개씩, 최대 14개 HWP
2. 전체 후보 문서 검증: `review_artifact_context`가 포함된 전체 후보 문서, 현재 pilot 기준 187개 문서

다른 컴퓨터에서 한컴오피스로 PDF를 저장한 뒤, 아래 검증 셀을 실행합니다.


In [11]:
import shutil

PDF_WORK_DIR = OUTPUTS_DIR / "pdf_validation_workflow"
REP_HWP_DIR = PDF_WORK_DIR / "representative_hwp_to_convert"
REP_PDF_DIR = PDF_WORK_DIR / "representative_manual_pdfs"
REP_MAP_CSV = PDF_WORK_DIR / "representative_pdf_conversion_map.csv"
REP_RESULT_CSV = OUTPUTS_DIR / "representative_pdf_artifact_validation.csv"

FULL_HWP_DIR = PDF_WORK_DIR / "full_hwp_to_convert"
FULL_PDF_DIR = PDF_WORK_DIR / "full_manual_pdfs"
FULL_MAP_CSV = PDF_WORK_DIR / "full_pdf_conversion_map.csv"
FULL_RESULT_CSV = OUTPUTS_DIR / "full_pdf_artifact_validation.csv"


def safe_filename(text: str, max_len: int = 120) -> str:
    text = re.sub(r'[<>:"/\\|?*]+', "_", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len]


def build_expected_pdf_name(doc_id: str, tokens: str, source_stem: str) -> str:
    return f"{doc_id}__tokens-{safe_filename(tokens, 45)}__{safe_filename(source_stem, 90)}.pdf"


def prepare_pdf_validation_files(
    review_detail: pd.DataFrame,
    mode: str,
    hwp_dir: Path,
    pdf_dir: Path,
    map_csv: Path,
) -> pd.DataFrame:
    """Copy HWP files to a conversion folder and create token-to-PDF mapping.

    mode:
    - representative: one representative source row per token
    - full: one HWP copy per unique source document, preserving all token mappings
    """
    hwp_dir.mkdir(parents=True, exist_ok=True)
    pdf_dir.mkdir(parents=True, exist_ok=True)

    mapping_columns = [
        "token",
        "source_file",
        "source_path",
        "doc_id",
        "manual_hwp_copy",
        "expected_pdf_name",
        "expected_pdf_path",
        "sample_context",
    ]
    if review_detail.empty:
        mapping = pd.DataFrame(columns=mapping_columns)
        mapping.to_csv(map_csv, index=False, encoding="utf-8-sig")
        print("mode:", mode)
        print("token rows: 0")
        print("unique tokens: 0")
        print("HWP files to convert: 0")
        print("HWP folder:", hwp_dir)
        print("PDF folder:", pdf_dir)
        print("mapping CSV:", map_csv)
        return mapping

    if mode == "representative":
        base = (
            review_detail
            .sort_values(["token", "count", "source_file"], ascending=[True, False, True])
            .drop_duplicates("token")
            .copy()
        )
        copy_plan = (
            base
            .groupby("source_path", as_index=False)
            .agg(
                source_file=("source_file", "first"),
                tokens=("token", lambda s: "_".join(sorted(set(map(str, s))))),
            )
            .sort_values("source_file")
            .reset_index(drop=True)
        )
        token_rows = base
    elif mode == "full":
        token_rows = review_detail.copy()
        copy_plan = (
            token_rows
            .groupby("source_path", as_index=False)
            .agg(
                source_file=("source_file", "first"),
                tokens=("token", lambda s: "_".join(sorted(set(map(str, s))))),
            )
            .sort_values("source_file")
            .reset_index(drop=True)
        )
    else:
        raise ValueError("mode must be 'representative' or 'full'")

    source_to_copy = {}
    for idx, row in copy_plan.iterrows():
        source_path = Path(row["source_path"])
        doc_id = f"D{idx + 1:03d}"
        pdf_name = build_expected_pdf_name(doc_id, row["tokens"], source_path.stem)
        hwp_name = pdf_name[:-4] + ".hwp"
        hwp_copy_path = hwp_dir / hwp_name

        if not hwp_copy_path.exists():
            shutil.copy2(source_path, hwp_copy_path)

        source_to_copy[str(source_path)] = {
            "doc_id": doc_id,
            "manual_hwp_copy": str(hwp_copy_path),
            "expected_pdf_name": pdf_name,
            "expected_pdf_path": str(pdf_dir / pdf_name),
        }

    rows = []
    for _, row in token_rows.iterrows():
        info = source_to_copy[str(Path(row["source_path"]))]
        rows.append(
            {
                "token": row["token"],
                "source_file": row["source_file"],
                "source_path": row["source_path"],
                "doc_id": info["doc_id"],
                "manual_hwp_copy": info["manual_hwp_copy"],
                "expected_pdf_name": info["expected_pdf_name"],
                "expected_pdf_path": info["expected_pdf_path"],
                "sample_context": row.get("sample_context", ""),
            }
        )

    mapping = pd.DataFrame(rows, columns=mapping_columns).sort_values(["doc_id", "token"])
    mapping.to_csv(map_csv, index=False, encoding="utf-8-sig")

    readme = map_csv.with_name(map_csv.stem + "_README.txt")
    readme.write_text(
        "\n".join(
            [
                "Manual PDF conversion guide",
                "",
                "1. Open each HWP in the hwp_to_convert folder with Hancom Office.",
                "2. Save/export as PDF into the paired manual_pdfs folder.",
                "3. Use the exact filename in expected_pdf_name.",
                "4. Re-run the validation cell in the notebook.",
                "",
                f"HWP folder: {hwp_dir}",
                f"PDF folder: {pdf_dir}",
                f"Mapping CSV: {map_csv}",
            ]
        ),
        encoding="utf-8",
    )

    print("mode:", mode)
    print("token rows:", len(mapping))
    print("unique tokens:", mapping["token"].nunique())
    print("HWP files to convert:", mapping["manual_hwp_copy"].nunique())
    print("HWP folder:", hwp_dir)
    print("PDF folder:", pdf_dir)
    print("mapping CSV:", map_csv)
    return mapping


def extract_pdf_text_safe(pdf_path: Path) -> tuple[str, str]:
    try:
        from pypdf import PdfReader
    except Exception as exc:
        return "", f"pypdf_import_failed: {exc!r}"

    try:
        reader = PdfReader(str(pdf_path))
        return "\n".join((page.extract_text() or "") for page in reader.pages), "ok"
    except Exception as exc:
        return "", f"pdf_text_extract_failed: {exc!r}"


def validate_manual_pdfs(map_csv: Path, result_csv: Path) -> pd.DataFrame:
    if not map_csv.exists():
        raise FileNotFoundError(f"PDF mapping CSV가 없습니다: {map_csv}")

    mapping = pd.read_csv(map_csv)
    result_columns = [
        "token",
        "source_file",
        "doc_id",
        "expected_pdf_path",
        "pdf_exists",
        "raw_contains",
        "clean_contains",
        "pdf_text_contains",
        "pdf_text_status",
        "sample_context",
        "decision",
        "reason",
    ]
    if mapping.empty:
        result = pd.DataFrame(columns=result_columns)
        result.to_csv(result_csv, index=False, encoding="utf-8-sig")
        print("mapping rows: 0")
        print("unique tokens: 0")
        print("unique docs: 0")
        print("existing PDFs: 0")
        print("result CSV:", result_csv)
        print("\n--- decision counts ---")
        print("no rows")
        return result

    rows = []
    hwp_text_cache = {}
    pdf_text_cache = {}

    for _, item in mapping.iterrows():
        token = str(item["token"])
        hwp_path = Path(item["source_path"])
        pdf_path = Path(item["expected_pdf_path"])

        hwp_key = str(hwp_path)
        if hwp_key not in hwp_text_cache:
            hwp_text_cache[hwp_key] = (
                extract_hwp_text_raw(hwp_path),
                extract_hwp_text(hwp_path),
            )
        raw_doc, clean_doc = hwp_text_cache[hwp_key]

        pdf_exists = pdf_path.exists()
        pdf_text = ""
        pdf_text_status = "pdf_missing"
        if pdf_exists:
            pdf_key = str(pdf_path)
            if pdf_key not in pdf_text_cache:
                pdf_text_cache[pdf_key] = extract_pdf_text_safe(pdf_path)
            pdf_text, pdf_text_status = pdf_text_cache[pdf_key]

        row = {
            "token": token,
            "source_file": item["source_file"],
            "doc_id": item["doc_id"],
            "expected_pdf_path": str(pdf_path),
            "pdf_exists": pdf_exists,
            "raw_contains": token in raw_doc["text"],
            "clean_contains": token in clean_doc["text"],
            "pdf_text_contains": token in pdf_text if pdf_text else False,
            "pdf_text_status": pdf_text_status,
            "sample_context": item.get("sample_context", ""),
        }

        if not pdf_exists:
            row["decision"] = "waiting_for_manual_pdf"
            row["reason"] = "PDF 파일이 아직 manual_pdfs 폴더에 없음"
        elif pdf_text_status != "ok":
            row["decision"] = "needs_ocr_or_manual"
            row["reason"] = "PDF 텍스트 추출 실패 또는 OCR 필요"
        elif row["pdf_text_contains"]:
            row["decision"] = "pdf_confirms_keep"
            row["reason"] = "PDF 텍스트에도 token이 존재함"
        else:
            row["decision"] = "pdf_suggests_artifact"
            row["reason"] = "raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼"

        rows.append(row)

    result = pd.DataFrame(rows, columns=result_columns)
    result.to_csv(result_csv, index=False, encoding="utf-8-sig")

    print("mapping rows:", len(mapping))
    print("unique tokens:", result["token"].nunique())
    print("unique docs:", result["source_file"].nunique())
    print("existing PDFs:", int(result["pdf_exists"].sum()))
    print("result CSV:", result_csv)
    print("\n--- decision counts ---")
    print(result["decision"].value_counts().to_string())

    return result


## 빠른 PDF token 검증 패치

전체 후보 PDF 검증이 오래 걸리는 문제를 줄이기 위해, PDF별로 token 존재 여부만 별도 Python 프로세스에서 확인합니다. 파일별 timeout을 두어 특정 PDF 하나가 전체 검증을 멈추지 않게 합니다.


In [12]:
# FAST_VALIDATION_PATCH
import json
import subprocess
import sys
from pathlib import Path

PDF_TOKEN_TIMEOUT_SECONDS = 12


def extract_pdf_token_flags_safe(pdf_path: Path, tokens: list[str], timeout_seconds: int = PDF_TOKEN_TIMEOUT_SECONDS) -> tuple[dict[str, bool], str]:
    """Check whether each token appears in PDF text, isolating slow PDFs in a subprocess."""
    unique_tokens = sorted(set(map(str, tokens)))
    empty = {token: False for token in unique_tokens}

    code = r"""
import json
import sys

pdf_path = sys.argv[1]
tokens = json.loads(sys.argv[2])
found = {token: False for token in tokens}

try:
    import fitz

    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text") or ""
            for token in tokens:
                if not found[token] and token in text:
                    found[token] = True
            if all(found.values()):
                break
    print(json.dumps({"status": "ok", "found": found}, ensure_ascii=True))
except Exception as fitz_exc:
    try:
        from pypdf import PdfReader

        reader = PdfReader(pdf_path)
        for page in reader.pages:
            text = page.extract_text() or ""
            for token in tokens:
                if not found[token] and token in text:
                    found[token] = True
            if all(found.values()):
                break
        print(json.dumps({"status": "ok", "found": found}, ensure_ascii=True))
    except Exception as exc:
        message = "fitz=" + repr(fitz_exc) + "; pypdf=" + repr(exc)
        print(json.dumps({"status": "pdf_text_extract_failed: " + message, "found": found}, ensure_ascii=True))
"""

    try:
        completed = subprocess.run(
            [sys.executable, "-c", code, str(pdf_path), json.dumps(unique_tokens, ensure_ascii=True)],
            capture_output=True,
            text=True,
            encoding="utf-8",
            timeout=timeout_seconds,
        )
    except subprocess.TimeoutExpired:
        return empty, f"pdf_text_extract_timeout_{timeout_seconds}s"

    stdout = (completed.stdout or "").strip()
    if completed.returncode != 0:
        stderr = (completed.stderr or "").strip()
        return empty, f"pdf_text_extract_failed: returncode={completed.returncode} stderr={stderr[:300]}"
    if not stdout:
        return empty, "pdf_text_extract_failed: empty_stdout"

    try:
        payload = json.loads(stdout.splitlines()[-1])
    except Exception as exc:
        return empty, f"pdf_text_extract_failed: bad_json {exc!r}"

    found = payload.get("found", {})
    status = payload.get("status", "ok")
    return {token: bool(found.get(token, False)) for token in unique_tokens}, status


def validate_manual_pdfs(map_csv: Path, result_csv: Path) -> pd.DataFrame:
    if not map_csv.exists():
        raise FileNotFoundError(f"PDF mapping CSV가 없습니다: {map_csv}")

    mapping = pd.read_csv(map_csv)
    result_columns = [
        "token",
        "source_file",
        "doc_id",
        "expected_pdf_path",
        "pdf_exists",
        "raw_contains",
        "clean_contains",
        "pdf_text_contains",
        "pdf_text_status",
        "sample_context",
        "decision",
        "reason",
    ]
    if mapping.empty:
        result = pd.DataFrame(columns=result_columns)
        result.to_csv(result_csv, index=False, encoding="utf-8-sig")
        print("mapping rows: 0")
        print("unique tokens: 0")
        print("unique docs: 0")
        print("existing PDFs: 0")
        print("result CSV:", result_csv)
        print("\n--- decision counts ---")
        print("no rows")
        return result

    rows = []
    hwp_text_cache = {}
    pdf_token_cache = {}

    pdf_groups = list(mapping.groupby("expected_pdf_path", sort=False))
    for pdf_idx, (pdf_path_text, pdf_group) in enumerate(pdf_groups, start=1):
        pdf_path = Path(pdf_path_text)
        pdf_exists = pdf_path.exists()
        tokens_for_pdf = sorted(set(map(str, pdf_group["token"])))

        if pdf_exists:
            pdf_key = str(pdf_path)
            if pdf_key not in pdf_token_cache:
                pdf_token_cache[pdf_key] = extract_pdf_token_flags_safe(pdf_path, tokens_for_pdf)
            pdf_token_flags, pdf_text_status = pdf_token_cache[pdf_key]
        else:
            pdf_token_flags = {token: False for token in tokens_for_pdf}
            pdf_text_status = "pdf_missing"

        if pdf_idx % 25 == 0 or pdf_idx == len(pdf_groups):
            print(f"validated PDFs {pdf_idx}/{len(pdf_groups)}")

        for _, item in pdf_group.iterrows():
            token = str(item["token"])
            hwp_path = Path(item["source_path"])

            hwp_key = str(hwp_path)
            if hwp_key not in hwp_text_cache:
                hwp_text_cache[hwp_key] = (
                    extract_hwp_text_raw(hwp_path),
                    extract_hwp_text(hwp_path),
                )
            raw_doc, clean_doc = hwp_text_cache[hwp_key]

            row = {
                "token": token,
                "source_file": item["source_file"],
                "doc_id": item["doc_id"],
                "expected_pdf_path": str(pdf_path),
                "pdf_exists": pdf_exists,
                "raw_contains": token in raw_doc["text"],
                "clean_contains": token in clean_doc["text"],
                "pdf_text_contains": bool(pdf_token_flags.get(token, False)),
                "pdf_text_status": pdf_text_status,
                "sample_context": item.get("sample_context", ""),
            }

            if not pdf_exists:
                row["decision"] = "waiting_for_manual_pdf"
                row["reason"] = "PDF 파일이 아직 manual_pdfs 폴더에 없음"
            elif pdf_text_status != "ok":
                row["decision"] = "needs_ocr_or_manual"
                row["reason"] = "PDF 텍스트 추출 실패 또는 OCR 필요"
            elif row["pdf_text_contains"]:
                row["decision"] = "pdf_confirms_keep"
                row["reason"] = "PDF 텍스트에도 token이 존재함"
            else:
                row["decision"] = "pdf_suggests_artifact"
                row["reason"] = "raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼"

            rows.append(row)

    result = pd.DataFrame(rows, columns=result_columns)
    result.to_csv(result_csv, index=False, encoding="utf-8-sig")

    print("mapping rows:", len(mapping))
    print("unique tokens:", result["token"].nunique())
    print("unique docs:", result["source_file"].nunique())
    print("existing PDFs:", int(result["pdf_exists"].sum()))
    print("result CSV:", result_csv)
    print("\n--- decision counts ---")
    print(result["decision"].value_counts().to_string())

    return result


## 1단계: 대표 문서 PDF 검증 준비

14개 token 각각에 대해 대표 문서를 1개씩 고릅니다. 같은 HWP에 여러 token이 있으면 HWP 복사본은 하나로 묶입니다.


In [ ]:
representative_pdf_map = prepare_pdf_validation_files(
    review_artifact_detail,
    mode="representative",
    hwp_dir=REP_HWP_DIR,
    pdf_dir=REP_PDF_DIR,
    map_csv=REP_MAP_CSV,
)

representative_pdf_map.head(5)


## 1단계: 대표 문서 PDF 저장 후 검증

`representative_hwp_to_convert`의 HWP를 한컴오피스에서 PDF로 저장한 뒤, PDF를 `representative_manual_pdfs`에 넣고 이 셀을 실행합니다.


In [14]:
representative_pdf_validation = validate_manual_pdfs(REP_MAP_CSV, REP_RESULT_CSV)
representative_pdf_validation[
    [
        "token",
        "source_file",
        "pdf_exists",
        "raw_contains",
        "clean_contains",
        "pdf_text_contains",
        "pdf_text_status",
        "decision",
        "reason",
    ]
]


validated PDFs 9/9
mapping rows: 14
unique tokens: 14
unique docs: 9
existing PDFs: 14
result CSV: C:\Users\goodm\Desktop\codeit\project_2nd\outputs\representative_pdf_artifact_validation.csv

--- decision counts ---
decision
pdf_suggests_artifact    12
pdf_confirms_keep         2


,token,source_file,pdf_exists,raw_contains,clean_contains,pdf_text_contains,pdf_text_status,decision,reason
0,汫,KOICA 전자조달_[지문] 에콰도르 국가 유전자원 데이터 은행 설립 및 역량.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
1,楴,경기도 고양시_고양시 도서관 홈페이지 온라인 결제시스템 구축 용역.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
2,潣,경기도 고양시_고양시 도서관 홈페이지 온라인 결제시스템 구축 용역.hwp,True,True,False,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
3,沤,경기도 군포시_(긴급)군포 AI기반 쓰레기 무단투기 단속시스템 구축 용역.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
4,爔,경기도 군포시_(긴급)군포 AI기반 쓰레기 무단투기 단속시스템 구축 용역.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
5,蕀,경기도 군포시_(긴급)군포 AI기반 쓰레기 무단투기 단속시스템 구축 용역.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
6,遠,경기도 군포시_(긴급)군포 AI기반 쓰레기 무단투기 단속시스템 구축 용역.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
7,浥,고려대학교_[재공지] 고려대학교 공간관리 통합시스템 구축 사업.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
8,牦,국립중앙의료원_(긴급)「2024년도 스마트의료지도시스템 고도화」.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
9,浵,"국민연금공단_(재공고)공동 연계급여정보시스템 재구축(1차, 2차).hwp",True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼


## 2단계: 전체 후보 문서 PDF 검증 준비

대표 문서 검증에서 방식이 유효하면, `review_artifact_context`가 포함된 전체 후보 문서를 대상으로 확장합니다.


In [ ]:
full_pdf_map = prepare_pdf_validation_files(
    review_artifact_detail,
    mode="full",
    hwp_dir=FULL_HWP_DIR,
    pdf_dir=FULL_PDF_DIR,
    map_csv=FULL_MAP_CSV,
)

full_pdf_map.head(5)


## 2단계: 전체 후보 문서 PDF 저장 후 검증

`full_hwp_to_convert`의 HWP를 PDF로 저장한 뒤, PDF를 `full_manual_pdfs`에 넣고 이 셀을 실행합니다.


In [16]:
full_pdf_validation = validate_manual_pdfs(FULL_MAP_CSV, FULL_RESULT_CSV)
full_pdf_validation[
    [
        "token",
        "source_file",
        "pdf_exists",
        "raw_contains",
        "clean_contains",
        "pdf_text_contains",
        "pdf_text_status",
        "decision",
        "reason",
    ]
].head(100)


validated PDFs 25/187


validated PDFs 50/187


validated PDFs 75/187


validated PDFs 100/187


validated PDFs 125/187


validated PDFs 150/187


validated PDFs 175/187


validated PDFs 187/187
mapping rows: 410
unique tokens: 14
unique docs: 187
existing PDFs: 409
result CSV: C:\Users\goodm\Desktop\codeit\project_2nd\outputs\full_pdf_artifact_validation.csv

--- decision counts ---
decision
pdf_suggests_artifact     400
pdf_confirms_keep           9
waiting_for_manual_pdf      1


,token,source_file,pdf_exists,raw_contains,clean_contains,pdf_text_contains,pdf_text_status,decision,reason
0,普,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
1,浵,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
2,浵,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
3,楴,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
4,汫,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
...,...,...,...,...,...,...,...,...,...
95,汫,경기도 성남시_성남시 통합성과관리시스템(BSC) 재구축 용역(협상에 의한.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
96,浵,경기도 성남시_성남시 통합성과관리시스템(BSC) 재구축 용역(협상에 의한.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
97,普,경기도 성남시_성남시정연구원 전산시스템(연구과제관리시스템)구축 용.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼
98,浵,경기도 성남시_성남시정연구원 전산시스템(연구과제관리시스템)구축 용.hwp,True,True,True,False,ok,pdf_suggests_artifact,raw/clean에는 있으나 PDF 텍스트에는 없어 artifact 가능성이 큼


## PDF 검증 결과 요약

대표/전체 검증 결과가 생성된 뒤 decision별 개수와 token별 판정을 요약합니다.


In [17]:
def summarize_pdf_validation(result: pd.DataFrame) -> pd.DataFrame:
    if result.empty:
        return pd.DataFrame()

    summary = (
        result
        .groupby("token", as_index=False)
        .agg(
            rows=("token", "size"),
            docs=("source_file", "nunique"),
            pdf_exists_count=("pdf_exists", "sum"),
            pdf_text_contains_count=("pdf_text_contains", "sum"),
            decisions=("decision", lambda s: ", ".join(sorted(set(map(str, s))))),
        )
        .sort_values(["pdf_text_contains_count", "docs", "token"], ascending=[False, False, True])
    )
    return summary


if "representative_pdf_validation" in globals():
    print("--- representative decision counts ---")
    print(representative_pdf_validation["decision"].value_counts().to_string())
    display(summarize_pdf_validation(representative_pdf_validation))

if "full_pdf_validation" in globals():
    print("--- full decision counts ---")
    print(full_pdf_validation["decision"].value_counts().to_string())
    display(summarize_pdf_validation(full_pdf_validation))


--- representative decision counts ---
decision
pdf_suggests_artifact    12
pdf_confirms_keep         2


,token,rows,docs,pdf_exists_count,pdf_text_contains_count,decisions
0,新,1,1,1,1,pdf_confirms_keep
11,舊,1,1,1,1,pdf_confirms_keep
1,普,1,1,1,0,pdf_suggests_artifact
2,楴,1,1,1,0,pdf_suggests_artifact
3,汫,1,1,1,0,pdf_suggests_artifact
4,沤,1,1,1,0,pdf_suggests_artifact
5,浥,1,1,1,0,pdf_suggests_artifact
6,浫,1,1,1,0,pdf_suggests_artifact
7,浵,1,1,1,0,pdf_suggests_artifact
8,潣,1,1,1,0,pdf_suggests_artifact


--- full decision counts ---
decision
pdf_suggests_artifact     400
pdf_confirms_keep           9
waiting_for_manual_pdf      1


,token,rows,docs,pdf_exists_count,pdf_text_contains_count,decisions
0,新,7,7,7,5,"pdf_confirms_keep, pdf_suggests_artifact"
11,舊,4,4,4,4,pdf_confirms_keep
7,浵,132,132,132,0,pdf_suggests_artifact
3,汫,85,85,85,0,pdf_suggests_artifact
2,楴,82,82,81,0,"pdf_suggests_artifact, waiting_for_manual_pdf"
1,普,60,60,60,0,pdf_suggests_artifact
6,浫,19,19,19,0,pdf_suggests_artifact
10,牦,4,4,4,0,pdf_suggests_artifact
4,沤,3,3,3,0,pdf_suggests_artifact
8,潣,3,3,3,0,pdf_suggests_artifact


<!-- artifact-validation-final-summary-2026-05-15 -->

## PDF 기반 artifact 검증 결론

PDF 1개(`D084__tokens-楴__경상북도 의성군_노지스마트농업시범사업 데이터플랫폼구축 정보화전.pdf`)는 변환 실패로 포기하고, 나머지 186개 PDF 기준으로 판단한다.

최종 검증 결과, 전체 410개 token 검증 row 중 `pdf_suggests_artifact`가 400개, `pdf_confirms_keep`가 9개, `waiting_for_manual_pdf`가 1개였다. `浵`, `汫`, `楴`, `普`, `浫` 등 주요 의심 token은 PDF 텍스트에서 확인되지 않아 HWP 제어 정보가 한자처럼 디코딩된 artifact로 보는 것이 타당하다.

반대로 정상 한자 확인용 positive control인 `新`, `舊`는 PDF에서도 확인되었다. 따라서 현재 cleaning 흐름은 한자를 무조건 삭제하는 것이 아니라, HWP artifact성 한자를 제거하고 실제 의미 있는 한자는 보존하는 방향으로 작동한다고 판단한다.
